# 🔍 Análise do Dataset DGA — Modelos Clássicos de Machine Learning

Este notebook realiza a análise do dataset **DGA (Domain Generation Algorithm)**, que contém domínios gerados por algoritmos maliciosos e domínios legítimos.

## Estrutura
1. Importações e Configurações
2. Carregamento e EDA (Análise Exploratória)
3. Feature Engineering
4. Pré-processamento
5. Modelos Clássicos: Naive Bayes, Decision Tree, Random Forest, KNN, SVM
6. Avaliação e Comparação

## 1. Importações e Configurações

In [5]:
import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd
from collections import Counter
import math
import time

# Visualização
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Pré-processamento e Feature Extraction
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# Modelos Clássicos
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

# Métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
    ConfusionMatrixDisplay
)

# Estilo dos gráficos
plt.style.use('dark_background')
sns.set_palette('husl')
COLORS = ['#00D4FF', '#FF6B6B', '#4ECDC4', '#FFE66D', '#A29BFE', '#FD79A8']

print('✅ Bibliotecas importadas com sucesso!')
print(f'   NumPy: {np.__version__}')
print(f'   Pandas: {pd.__version__}')

✅ Bibliotecas importadas com sucesso!
   NumPy: 2.2.6
   Pandas: 2.3.3


## 2. Carregamento e EDA (Análise Exploratória)

In [ ]:
# Carregamento do dataset
df = pd.read_csv('dga_data.csv')

print('=' * 60)
print('📊 INFORMAÇÕES DO DATASET')
print('=' * 60)
print(f'  Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'  Colunas: {df.columns.tolist()}')
print(f'  Memória: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')
print()
print('Tipos de dados:')
print(df.dtypes)
print()
print('Valores nulos:')
print(df.isnull().sum())
print()
df.head(10)

In [ ]:
# Distribuição de classes
label_counts = df['isDGA'].value_counts()
print('\n📌 Distribuição de Classes (isDGA):')
for k, v in label_counts.items():
    pct = v / len(df) * 100
    print(f'  {k}: {v:,} ({pct:.1f}%)')

print(f'\n📌 Subclasses de malware: {df["subclass"].nunique()} únicas')
print(df['subclass'].value_counts().head(15))

In [ ]:
# ── Visualizações de EDA ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0D1117')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Distribuição de classes (pizza)
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('#161B22')
wedge_colors = [COLORS[1], COLORS[0]]
wedges, texts, autotexts = ax1.pie(
    label_counts.values,
    labels=label_counts.index,
    autopct='%1.1f%%',
    colors=wedge_colors,
    startangle=140,
    wedgeprops={'edgecolor': '#0D1117', 'linewidth': 2}
)
for at in autotexts:
    at.set_color('white')
    at.set_fontsize(11)
ax1.set_title('Distribuição isDGA', color='white', fontsize=13, fontweight='bold')

# 2. Top 15 subclasses
ax2 = fig.add_subplot(gs[0, 1:])
ax2.set_facecolor('#161B22')
top_sub = df['subclass'].value_counts().head(15)
bars = ax2.barh(range(len(top_sub)), top_sub.values, color=COLORS[0], alpha=0.85)
ax2.set_yticks(range(len(top_sub)))
ax2.set_yticklabels(top_sub.index, color='white', fontsize=8)
ax2.set_xlabel('Contagem', color='#8B949E')
ax2.set_title('Top 15 Subclasses de Malware', color='white', fontsize=13, fontweight='bold')
ax2.tick_params(colors='#8B949E')
ax2.spines[['top','right','left','bottom']].set_color('#30363D')
for bar, val in zip(bars, top_sub.values):
    ax2.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', color='#8B949E', fontsize=7)

# 3. Comprimento de domínios
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor('#161B22')
for label, color in zip(['legit', 'dga'], [COLORS[0], COLORS[1]]):
    subset = df[df['isDGA'] == label]['domain'].dropna().str.len()
    ax3.hist(subset, bins=40, alpha=0.6, label=label, color=color)
ax3.set_xlabel('Comprimento do Domínio', color='#8B949E')
ax3.set_ylabel('Frequência', color='#8B949E')
ax3.set_title('Comprimento dos Domínios', color='white', fontsize=13, fontweight='bold')
ax3.legend(facecolor='#161B22', labelcolor='white')
ax3.tick_params(colors='#8B949E')
ax3.spines[['top','right','left','bottom']].set_color('#30363D')

# 4. Proporção de dígitos
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor('#161B22')
for label, color in zip(['legit', 'dga'], [COLORS[0], COLORS[1]]):
    subset = df[df['isDGA'] == label]['domain']
    ratio = subset.dropna().apply(lambda x: sum(c.isdigit() for c in str(x)) / max(len(str(x)), 1))
    ax4.hist(ratio, bins=30, alpha=0.6, label=label, color=color)
ax4.set_xlabel('Proporção de Dígitos', color='#8B949E')
ax4.set_ylabel('Frequência', color='#8B949E')
ax4.set_title('Proporção de Dígitos', color='white', fontsize=13, fontweight='bold')
ax4.legend(facecolor='#161B22', labelcolor='white')
ax4.tick_params(colors='#8B949E')
ax4.spines[['top','right','left','bottom']].set_color('#30363D')

# 5. Exemplos de domínios
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor('#161B22')
ax5.axis('off')

legit_examples = df[df['isDGA'] == 'legit']['domain'].sample(5, random_state=42).tolist()
dga_examples = df[df['isDGA'] == 'dga']['domain'].sample(5, random_state=42).tolist()
text_content = '🌐 Legítimos:\n\n' + '\n'.join(f'  {d}' for d in legit_examples)
text_content += '\n\n🦠 DGA:\n\n' + '\n'.join(f'  {d[:20]}...' if len(d) > 20 else f'  {d}' for d in dga_examples)
ax5.text(0.05, 0.95, text_content, transform=ax5.transAxes,
         va='top', color='white', fontsize=9,
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#21262D', edgecolor='#30363D'))
ax5.set_title('Exemplos de Domínios', color='white', fontsize=13, fontweight='bold')

fig.suptitle('DGA Dataset — Análise Exploratória', color='white',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('eda_overview.png', bbox_inches='tight', facecolor='#0D1117', dpi=120)
plt.show()
print('✅ EDA concluída!')

## 3. Feature Engineering

Extraímos **features numéricas** a partir dos nomes de domínio:

| Feature | Descrição |
|---|---|
| `length` | Comprimento do domínio |
| `entropy` | Entropia de Shannon (aleatoriedade) |
| `vowel_ratio` | Proporção de vogais |
| `digit_ratio` | Proporção de dígitos |
| `consonant_ratio` | Proporção de consoantes |
| `unique_chars` | Número de caracteres únicos |
| `max_consec_consonants` | Máximo de consoantes consecutivas |
| `has_hyphen` | Presença de hífen |
| `has_digits` | Presença de dígitos |
| `n_gram_tfidf` | TF-IDF de trigramas de caracteres |

In [ ]:
def shannon_entropy(s):
    """Calcula a entropia de Shannon de uma string."""
    if len(s) == 0:
        return 0.0
    counts = Counter(s)
    total = len(s)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

def max_consecutive_consonants(s):
    """Máximo de consoantes consecutivas (indício de palavra artificial)."""
    consonants = set('bcdfghjklmnpqrstvwxyz')
    max_c = curr_c = 0
    for ch in s.lower():
        if ch in consonants:
            curr_c += 1
            max_c = max(max_c, curr_c)
        else:
            curr_c = 0
    return max_c

def extract_features(df):
    """Extrai features numéricas de domínios."""
    domains = df['domain'].str.lower()
    vowels = set('aeiou')
    consonants = set('bcdfghjklmnpqrstvwxyz')

    features = pd.DataFrame()
    features['length'] = domains.str.len()
    features['entropy'] = domains.apply(shannon_entropy)
    features['vowel_ratio'] = domains.apply(
        lambda x: sum(c in vowels for c in str(x)) / max(len(str(x)), 1) if pd.notna(x) else 0
    )
    features['digit_ratio'] = domains.apply(
        lambda x: sum(c.isdigit() for c in str(x)) / max(len(str(x)), 1) if pd.notna(x) else 0
    )
    features['consonant_ratio'] = domains.apply(
        lambda x: sum(c in consonants for c in str(x)) / max(len(str(x)), 1) if pd.notna(x) else 0
    )
    features['unique_chars'] = domains.apply(lambda x: len(set(str(x))) if pd.notna(x) else 0)
    features['max_consec_consonants'] = domains.apply(max_consecutive_consonants)
    features['has_hyphen'] = domains.str.contains('-').astype(int)
    features['has_digits'] = domains.str.contains(r'\d').astype(int)
    features['length_entropy'] = features['length'] * features['entropy']  # feature composta
    return features

print('⚙️ Extraindo features numéricas...')
t0 = time.time()
num_features = extract_features(df)
print(f'   ✅ {num_features.shape[1]} features extraídas em {time.time()-t0:.1f}s')

print('\n⚙️ Extraindo TF-IDF de trigramas de caracteres...')
t0 = time.time()
tfidf = TfidfVectorizer(analyzer='char', ngram_range=(2, 3), max_features=200, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df['domain'].str.lower())
print(f'   ✅ TF-IDF shape: {X_tfidf.shape} em {time.time()-t0:.1f}s')

num_features.describe()

In [ ]:
# Correlação entre features numéricas
fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#161B22')

corr = num_features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='coolwarm', center=0,
    annot=True, fmt='.2f', annot_kws={'size': 9},
    linewidths=0.5, linecolor='#30363D',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlação entre Features Numéricas', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('feature_correlation.png', bbox_inches='tight', facecolor='#0D1117', dpi=120)
plt.show()

In [ ]:
# Boxplots: Entropia e Comprimento por classe
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0D1117')

# Reset index para garantir alinhamento limpo após eventuais NaNs truncados fora
df_clean = df.reset_index(drop=True)
feat_clean = num_features.reset_index(drop=True)
plot_df = feat_clean[['entropy', 'length']].copy()
plot_df['label'] = df_clean['isDGA']

for ax, feat, title in zip(axes, ['entropy', 'length'], ['Entropia de Shannon', 'Comprimento']):
    ax.set_facecolor('#161B22')
    
    # Extract clean vectors skipping any remaining NaNs safely
    d_legit = plot_df[plot_df['label'] == 'legit'][feat].dropna().values
    d_dga = plot_df[plot_df['label'] == 'dga'][feat].dropna().values
    data_by_class = [d_legit, d_dga]
    
    bp = ax.boxplot(data_by_class, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2),
                    flierprops=dict(marker='o', markersize=2, alpha=0.3))
    for patch, color in zip(bp['boxes'], [COLORS[0], COLORS[1]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Legítimo', 'DGA'], color='white')
    ax.set_title(title, color='white', fontsize=13, fontweight='bold')
    ax.tick_params(colors='#8B949E')
    ax.spines[['top','right','left','bottom']].set_color('#30363D')
    ax.set_facecolor('#161B22')

fig.suptitle('Features por Classe', color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('features_by_class.png', bbox_inches='tight', facecolor='#0D1117', dpi=120)
plt.show()


## 4. Pré-processamento

In [ ]:
# Codificação do target
le = LabelEncoder()
y = le.fit_transform(df['isDGA'])  # dga=0, legit=1 (ordem alfabética)
print(f'Classes: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Combinar features numéricas + TF-IDF
X_num_scaled = StandardScaler().fit_transform(num_features)
X_combined = hstack([csr_matrix(X_num_scaled), X_tfidf])
print(f'\nMatrix combinada: {X_combined.shape}')

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y,
    test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape[0]:,} amostras | Test: {X_test.shape[0]:,} amostras')

# Distribuição no split
train_dga = y_train.sum()
test_dga = y_test.sum()
print(f'Train — DGA: {len(y_train)-train_dga:,} ({(len(y_train)-train_dga)/len(y_train)*100:.1f}%)')
print(f'       Legit: {train_dga:,} ({train_dga/len(y_train)*100:.1f}%)')

## 5. Treinamento dos Modelos Clássicos

In [ ]:
# Definição dos modelos
models = {
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=15, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'SVM (Linear)': LinearSVC(max_iter=2000, random_state=42),
}

results = {}
trained_models = {}

print('🚀 Treinando modelos...')
print('=' * 65)

for name, model in models.items():
    t0 = time.time()
    print(f'  ⏳ {name}...', end=' ', flush=True)

    # Naive Bayes e KNN precisam de array denso
    if name in ['Naive Bayes', 'KNN (k=5)']:
        X_tr = X_train.toarray()
        X_te = X_test.toarray()
    else:
        X_tr = X_train
        X_te = X_test

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    train_time = time.time() - t0

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Tempo (s)': train_time
    }
    trained_models[name] = (model, y_pred, X_te)

    print(f'✅  Acc={acc:.4f}  F1={f1:.4f}  [{train_time:.1f}s]')

print('='*65)
print('\n📊 Tabela Resumo:')
results_df = pd.DataFrame(results).T.sort_values('F1-Score', ascending=False)
results_df[['Accuracy','Precision','Recall','F1-Score']] = results_df[['Accuracy','Precision','Recall','F1-Score']].applymap(lambda x: f'{x:.4f}')
results_df['Tempo (s)'] = results_df['Tempo (s)'].apply(lambda x: f'{float(x):.1f}s')
results_df

## 6. Avaliação e Comparação Visual

In [ ]:
# Gráfico comparativo de métricas
res_numeric = pd.DataFrame({k: v for k, v in [(n, d) for n, d in
    [(name, {mk: mv for mk, mv in m.items() if mk != 'Tempo (s)'})
     for name, m in [(n2, m2) for n2, m2 in
     [(name3, {mk3: float(mv3) for mk3, mv3 in model3.items()})
      for name3, model3 in [
         (name_i, {mk: float(mv) for mk, mv in {
             'Accuracy': accuracy_score(y_test, trained_models[name_i][1]),
             'Precision': precision_score(y_test, trained_models[name_i][1], average='weighted'),
             'Recall': recall_score(y_test, trained_models[name_i][1], average='weighted'),
             'F1-Score': f1_score(y_test, trained_models[name_i][1], average='weighted'),
         }.items()})
         for name_i in models.keys()
      ]
     ]
    ]
}]}).T

# Simplificado: reconstruir results com valores float
results_float = {}
for name in models.keys():
    y_pred_i = trained_models[name][1]
    results_float[name] = {
        'Accuracy': accuracy_score(y_test, y_pred_i),
        'Precision': precision_score(y_test, y_pred_i, average='weighted'),
        'Recall': recall_score(y_test, y_pred_i, average='weighted'),
        'F1-Score': f1_score(y_test, y_pred_i, average='weighted'),
    }

rf_df = pd.DataFrame(results_float).T

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#161B22')

x = np.arange(len(rf_df))
width = 0.2
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for i, (metric, color) in enumerate(zip(metrics_to_plot, COLORS)):
    bars = ax.bar(x + i*width, rf_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(rf_df.index, rotation=15, ha='right', color='white', fontsize=10)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Score', color='#8B949E')
ax.set_title('Comparação de Métricas — Modelos Clássicos', color='white', fontsize=14, fontweight='bold')
ax.legend(facecolor='#21262D', labelcolor='white', framealpha=0.9)
ax.tick_params(colors='#8B949E')
ax.spines[['top','right','left','bottom']].set_color('#30363D')
ax.yaxis.grid(True, color='#30363D', alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('metrics_comparison.png', bbox_inches='tight', facecolor='#0D1117', dpi=120)
plt.show()

In [ ]:
# Matrizes de confusão — todos os modelos
n_models = len(models)
n_cols = 3
n_rows = math.ceil(n_models / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 5))
fig.patch.set_facecolor('#0D1117')
axes_flat = axes.flatten()

class_names = le.classes_

for idx, (name, (model, y_pred_m, _)) in enumerate(trained_models.items()):
    ax = axes_flat[idx]
    ax.set_facecolor('#161B22')
    cm = confusion_matrix(y_test, y_pred_m)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, color='white', fontsize=13, fontweight='bold')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.tick_params(colors='white')
    for text in ax.texts:
        text.set_color('white')
        text.set_fontsize(11)

# Esconder eixos extras
for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle('Matrizes de Confusão — Modelos Clássicos', color='white',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight', facecolor='#0D1117', dpi=120)
plt.show()

In [ ]:
# Curvas ROC (modelos com predict_proba ou decision_function)
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#161B22')

for idx, (name, (model, y_pred_m, X_te_m)) in enumerate(trained_models.items()):
    try:
        if hasattr(model, 'predict_proba'):
            y_score = model.predict_proba(X_te_m)[:, 1]
        elif hasattr(model, 'decision_function'):
            y_score = model.decision_function(X_te_m)
        else:
            continue
        fpr, tpr, _ = roc_curve(y_test, y_score)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=COLORS[idx % len(COLORS)], lw=2,
                label=f'{name} (AUC={roc_auc:.3f})')
    except Exception as e:
        print(f'⚠️ ROC indisponível para {name}: {e}')

ax.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.5, label='Aleatório (AUC=0.500)')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])
ax.set_xlabel('Taxa de Falso Positivo', color='#8B949E', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiro Positivo', color='#8B949E', fontsize=12)
ax.set_title('Curvas ROC — Modelos Clássicos', color='white', fontsize=14, fontweight='bold')
ax.legend(facecolor='#21262D', labelcolor='white', framealpha=0.9, fontsize=10)
ax.tick_params(colors='#8B949E')
ax.spines[['top','right','left','bottom']].set_color('#30363D')
ax.yaxis.grid(True, color='#30363D', alpha=0.5)
ax.xaxis.grid(True, color='#30363D', alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight', facecolor='#0D1117', dpi=120)
plt.show()

In [ ]:
# Feature Importance (Random Forest)
rf_model = trained_models['Random Forest'][0]
feature_names = list(num_features.columns) + [f'tfidf_{v}' for v in tfidf.get_feature_names_out()]
importances = rf_model.feature_importances_

# Top 20 features
top_idx = np.argsort(importances)[::-1][:20]
top_names = [feature_names[i] for i in top_idx]
top_vals = importances[top_idx]

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#161B22')

colors_bar = [COLORS[0] if not n.startswith('tfidf') else COLORS[2] for n in top_names]
ax.barh(range(len(top_names)), top_vals[::-1], color=colors_bar[::-1], alpha=0.85)
ax.set_yticks(range(len(top_names)))
ax.set_yticklabels(top_names[::-1], color='white', fontsize=9)
ax.set_xlabel('Importância', color='#8B949E')
ax.set_title('Top 20 Features — Random Forest', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='#8B949E')
ax.spines[['top','right','left','bottom']].set_color('#30363D')
ax.xaxis.grid(True, color='#30363D', alpha=0.5)
ax.set_axisbelow(True)

# Legenda de cores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS[0], label='Feature Numérica'),
    Patch(facecolor=COLORS[2], label='Feature TF-IDF')
]
ax.legend(handles=legend_elements, facecolor='#21262D', labelcolor='white')

plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', facecolor='#0D1117', dpi=120)
plt.show()

print('\n✅ Análise Clássica Concluída!')
print('\n📋 Relatório Detalhado — Melhor Modelo:')
best_model_name = max(results_float, key=lambda x: results_float[x]['F1-Score'])
print(f'   Melhor Modelo: {best_model_name}')
print(classification_report(y_test, trained_models[best_model_name][1], target_names=class_names))